# Auto-label every answer: grammatically correct or not
Reads `merged_for_annotation.csv` (from the merge notebook), scores **every answer** with a grammar-acceptability model (CoLA), and writes new `auto_grammar_*` columns.
Your manual column `is_grammatically_correct` is left untouched (unless you set `PREFILL_MANUAL = True`), so you can review and correct the model.

**Runtime tip:** Runtime → Change runtime type → T4 GPU makes this much faster (CPU also works for a few thousand rows).

In [ ]:
!pip install -q -U transformers pandas openpyxl

## 1. Config

In [ ]:
INPUT_CSV  = "merged_for_annotation.csv"   # upload this file to Colab (left sidebar -> Files), or reuse the one from the merge notebook
OUT_CSV    = "merged_grammar_labeled.csv"
OUT_XLSX   = "merged_grammar_labeled.xlsx"

MODEL_NAME = "textattack/roberta-base-CoLA"   # fine-tuned on CoLA (linguistic acceptability)
THRESHOLD  = 0.5          # score >= THRESHOLD  ->  "Yes" (grammatically correct)
REVIEW_BAND = (0.3, 0.7)  # scores inside this band are flagged for human review

# Answers like "Denmark" or "1998" are fragments, not sentences, so a grammar model can't judge them fairly.
SHORT_MAX_WORDS = 3
SHORT_POLICY = "na"       # "na"    -> label short fragments "N/A"
                          # "check" -> run the model on them anyway (noisy!)

USE_LANGUAGETOOL = False  # True = also run LanguageTool rule checker (needs Java, slower) and list the issues found
PREFILL_MANUAL   = False  # True = also copy the auto label into your manual `is_grammatically_correct` column
BATCH_SIZE = 64

## 2. Load the merged data

In [ ]:
import pandas as pd, os
if not os.path.exists(INPUT_CSV):
    from google.colab import files
    print("Upload merged_for_annotation.csv ...")
    up = files.upload()
    INPUT_CSV = list(up.keys())[0]

df = pd.read_csv(INPUT_CSV, dtype=str, keep_default_na=False)
print(df.shape)
df[["id", "source", "answer"]].head()

## 3. Load the grammar model and verify its label direction

In [ ]:
import torch, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device).eval()

@torch.no_grad()
def p_label1(texts, batch_size=BATCH_SIZE):
    """Probability of LABEL_1 for each text."""
    out = []
    for i in range(0, len(texts), batch_size):
        enc = tok(texts[i:i + batch_size], padding=True, truncation=True,
                  max_length=128, return_tensors="pt").to(device)
        out.extend(torch.softmax(model(**enc).logits, dim=-1)[:, 1].cpu().tolist())
    return out

# Sanity check: figure out whether LABEL_1 means "acceptable" (sources disagree, so we test it).
good = ["She went to the market yesterday and bought some fresh vegetables.",
        "The committee approved the proposal after a long discussion.",
        "My brother lives in London, and he works at a hospital."]
bad  = ["She go to market yesterday buyed vegetable fresh the.",
        "Committee the proposal approved after discussion long a.",
        "My brother live in London and he work at hospital the."]
g, b = np.mean(p_label1(good)), np.mean(p_label1(bad))
ACCEPTABLE_IS_LABEL1 = g > b
print(f"mean P(LABEL_1): good={g:.2f}  bad={b:.2f}  ->  acceptable = LABEL_{1 if ACCEPTABLE_IS_LABEL1 else 0}")
assert abs(g - b) > 0.2, "Model can't separate the sanity sentences - try another MODEL_NAME"

def p_acceptable(texts):
    p1 = np.array(p_label1(texts))
    return p1 if ACCEPTABLE_IS_LABEL1 else 1 - p1

## 4. Score every answer

In [ ]:
import re

def clean(ans):
    # AmbigQA multi-answers were joined with " | "; treat them as a list
    return re.sub(r"\s*\|\s*", ", ", str(ans)).strip()

def split_sentences(text):
    parts = re.split(r"(?<=[.!?])\s+", text)
    return [p for p in parts if p.strip()]

def n_words(text):
    return len(text.split())

answers = df["answer"].tolist()
cleaned = [clean(a) for a in answers]

# Decide which answers get scored
answer_type, to_score = [], {}
for i, t in enumerate(cleaned):
    if t == "":
        answer_type.append("empty")
    elif n_words(t) <= SHORT_MAX_WORDS and not re.search(r"[.!?]$", t):
        answer_type.append("short_span")
        if SHORT_POLICY == "check":
            to_score[i] = [t]
    else:
        answer_type.append("sentence")
        to_score[i] = split_sentences(t) or [t]

# Score each unique sentence once
unique_sents = sorted({s for sents in to_score.values() for s in sents})
print(f"Scoring {len(unique_sents)} unique sentences from {len(to_score)} answers ...")
probs = dict(zip(unique_sents, p_acceptable(unique_sents))) if unique_sents else {}

# Aggregate per answer
mean_score, min_score = [None] * len(df), [None] * len(df)
for i, sents in to_score.items():
    ps = [probs[s] for s in sents]
    mean_score[i], min_score[i] = float(np.mean(ps)), float(np.min(ps))

def label_for(i):
    if answer_type[i] == "empty":
        return "N/A"
    if answer_type[i] == "short_span" and SHORT_POLICY == "na":
        return "N/A"
    return "Yes" if mean_score[i] >= THRESHOLD else "No"

df["auto_grammar_answer_type"] = answer_type
df["auto_grammar_score"] = [f"{s:.3f}" if s is not None else "" for s in mean_score]
df["auto_grammar_min_sentence_score"] = [f"{s:.3f}" if s is not None else "" for s in min_score]
df["auto_grammar_label"] = [label_for(i) for i in range(len(df))]
df["auto_grammar_needs_review"] = [
    "Yes" if (mean_score[i] is not None and REVIEW_BAND[0] <= mean_score[i] <= REVIEW_BAND[1]) else ""
    for i in range(len(df))
]
df["auto_grammar_issues"] = ""

print(df["auto_grammar_label"].value_counts())

## 5. (Optional) LanguageTool: list the actual grammar issues

In [ ]:
if USE_LANGUAGETOOL:
    import subprocess, sys
    subprocess.run("apt-get install -y -qq default-jre-headless > /dev/null", shell=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "language-tool-python"])
    import language_tool_python
    tool = language_tool_python.LanguageTool("en-US")
    IGNORE_CATEGORIES = {"TYPOS", "CASING", "STYLE", "PUNCTUATION", "TYPOGRAPHY", "REDUNDANCY"}  # names/spans trigger many false alarms

    issues = []
    for i, t in enumerate(cleaned):
        if answer_type[i] != "sentence":
            issues.append("")
            continue
        try:
            ms = [m for m in tool.check(t) if str(getattr(m, "category", "")).upper() not in IGNORE_CATEGORIES]
        except Exception as e:
            ms = []
        issues.append("; ".join(f"{getattr(m, 'ruleId', '?')}: {getattr(m, 'message', '')}" for m in ms))
    df["auto_grammar_issues"] = issues

    # An answer only counts as correct if the model AND the rule checker agree
    mask = (df["auto_grammar_label"] == "Yes") & (df["auto_grammar_issues"] != "")
    df.loc[mask, "auto_grammar_label"] = "No"
    df.loc[mask, "auto_grammar_needs_review"] = "Yes"
    print("Downgraded by LanguageTool:", int(mask.sum()))
    print(df["auto_grammar_label"].value_counts())
else:
    print("Skipped (USE_LANGUAGETOOL = False)")

## 6. Inspect results

In [ ]:
print(pd.crosstab(df["source"], df["auto_grammar_label"]))
print()
print("Needs human review:", int((df["auto_grammar_needs_review"] == "Yes").sum()))

pd.set_option("display.max_colwidth", 120)
print("\n--- Examples labeled NO ---")
display(df[df["auto_grammar_label"] == "No"][["source", "answer", "auto_grammar_score"]].head(10))
print("--- Examples labeled YES ---")
display(df[df["auto_grammar_label"] == "Yes"][["source", "answer", "auto_grammar_score"]].head(10))

## 7. Save (CSV + Excel with dropdowns) and download

In [ ]:
if PREFILL_MANUAL:
    m = df["auto_grammar_label"].isin(["Yes", "No"])
    df.loc[m, "is_grammatically_correct"] = df.loc[m, "auto_grammar_label"]

df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

label_cols = [c for c in ["is_grammatically_correct", "is_ambiguous", "is_hallucinated"] if c in df.columns]
to_fill = label_cols + ["ambiguity_sources"]
auto_cols = [c for c in df.columns if c.startswith("auto_")]
widths = {"context": 60, "question": 40, "answer": 40, "ref_ambigqa_disambiguated": 40,
          "ambiguity_sources": 35, "notes": 30, "auto_grammar_issues": 40}

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name="annotate")
    ws = writer.sheets["annotate"]
    cols = list(df.columns)
    last = len(df) + 1

    dv = DataValidation(type="list", formula1='"Yes,No"', allow_blank=True)
    ws.add_data_validation(dv)
    for c in label_cols:
        L = get_column_letter(cols.index(c) + 1)
        dv.add(f"{L}2:{L}{last}")

    yellow = PatternFill("solid", fgColor="FFF2CC")   # you fill these
    blue   = PatternFill("solid", fgColor="DDEBF7")   # model-generated
    for idx, name in enumerate(cols, start=1):
        cell = ws.cell(row=1, column=idx)
        cell.font = Font(bold=True)
        if name in to_fill:   cell.fill = yellow
        if name in auto_cols: cell.fill = blue
        ws.column_dimensions[get_column_letter(idx)].width = widths.get(name, 18)

    for row in ws.iter_rows(min_row=2, max_row=last):
        for cell in row:
            cell.alignment = Alignment(wrap_text=True, vertical="top")
    ws.freeze_panes = "D2"

from google.colab import files
files.download(OUT_XLSX)
files.download(OUT_CSV)